# 9주차. 몬테카를로 시뮬레이션

## 오늘의 목표

이번 시간에는 확률분포를 직접 유도하지 않는다. 대신 기획자가 **불확실한 입력값을 분포로 표현하고**, AI에게 시뮬레이션 코드를 만들게 한 뒤, 결과를 의사결정 언어로 해석하는 연습을 한다.

수업이 끝나면 다음을 할 수 있어야 한다.

1. 점 추정과 분포 추정의 차이를 한 문장으로 설명한다.
2. 입력 변수에 적절한 분포를 붙인다.
3. AI에게 몬테카를로 시뮬레이션 코드를 요청한다.
4. 평균, 90% 구간, 손실 확률을 해석한다.
5. 핵심 리스크 변수를 찾아 의사결정 권고로 연결한다.


## 1. 점 추정 vs 분포 추정

점 추정은 하나의 숫자로 계획을 말한다.

> 이 사업의 5년 순이익은 10억 원이다.

분포 추정은 가능한 범위를 함께 말한다.

> 평균은 10억 원이지만, 나쁜 경우에는 -2억 원까지 내려갈 수 있고, 손실 확률은 18%다.

기획에서 중요한 것은 평균 하나가 아니라 **나쁜 경우가 얼마나 나쁜가**, **손실 가능성이 의사결정자가 감당할 수 있는가**이다.

| 질문 | 점 추정 | 몬테카를로 시뮬레이션 |
|---|---:|---:|
| 결과 표현 | 숫자 1개 | 결과 분포 |
| 불확실성 | 숨겨짐 | 드러남 |
| 의사결정 기준 | 평균 | 구간, 손실 확률, 임계값 초과 확률 |


## 2. 분포 4종 매뉴얼

분포 선택은 완벽한 수학 문제가 아니라 **입력값에 대해 무엇을 알고 있는가**를 정리하는 일이다.

| 분포 | 언제 쓰는가 | AI에게 줄 정보 |
|---|---|---|
| 정규분포 | 평균 주변으로 대칭적으로 흔들릴 때 | 평균, 표준편차 |
| 삼각분포 | 최소·최빈·최대만 대략 알 때 | 최소, 최빈, 최대 |
| 균등분포 | 범위만 알고 어느 값이 더 그럴듯한지 모를 때 | 최소, 최대 |
| 로그정규분포 | 0보다 크고 큰 값 쪽 꼬리가 길 때 | 중앙값 또는 평균, 변동성 |

**기획자의 원칙**

- 매출, 비용, 수요처럼 0보다 작을 수 없는 값은 음수가 나오지 않게 요청한다.
- 전문가 추정은 삼각분포로 시작하면 설명하기 쉽다.
- 데이터가 없으면 분포를 꾸며내기보다 "범위만 안다"고 두고 균등분포로 시작한다.


---

## 3. 사례: 청년 일자리 정책 확대 여부

**배경**

한 지자체가 청년 일자리 매칭 사업을 5년 동안 확대하려 한다. 담당자는 단일 예산안보다, 불확실성을 반영한 손익 범위를 보고 싶어 한다.

**결정 질문**

> 5년 누적 순편익이 0보다 클 가능성이 충분히 높은가? 손실 위험을 감수하고 확대할 것인가?

**결과 변수**

- 5년 누적 순편익 = 총 사회적 편익 - 총 운영비

**초기 입력 가정**

| 입력 변수 | 의미 | 분포 초안 |
|---|---|---|
| 연간 참여자 수 | 매년 사업에 참여하는 청년 수 | 삼각(800, 1000, 1400) |
| 1인당 순편익 | 취업 성공, 소득 증가, 사회적 비용 절감의 합 | 삼각(100만, 180만, 300만) |
| 연간 고정 운영비 | 인건비·시스템·홍보비 | 정규(12억, 2억) |
| 성과 하락률 | 시간이 지나며 효과가 줄어드는 비율 | 균등(0%, 20%) |

이 표는 정답이 아니라 AI와 대화하기 위한 출발점이다.


### 1단계. 입력 변수와 분포 점검

AI에게 먼저 코드가 아니라 **모델 구조가 타당한지** 점검하게 한다.

**AI에게 이렇게 질문하세요**

```text
나는 청년 일자리 정책의 5년 누적 순편익을 몬테카를로 시뮬레이션하려고 해.
결과 변수는 총 사회적 편익 - 총 운영비야.
입력 변수 초안은 아래와 같아.
- 연간 참여자 수: 삼각(800, 1000, 1400)
- 1인당 순편익: 삼각(100만, 180만, 300만)
- 연간 고정 운영비: 정규(12억, 2억)
- 성과 하락률: 균등(0%, 20%)

이 입력 변수와 분포 선택이 기획 의사결정에 적절한지 검토해줘.
빠진 변수, 음수가 나오면 안 되는 변수, 분포를 바꾸는 게 나은 변수를 표로 정리해줘.
```


**AI의 답변 예시**

| 항목 | 검토 결과 | 수정 제안 |
|---|---|---|
| 연간 참여자 수 | 최소·최빈·최대 추정이 있으므로 삼각분포 적절 | 정수로 샘플링 |
| 1인당 순편익 | 양수이고 큰 값 가능성이 있으므로 삼각 또는 로그정규 가능 | 수업에서는 삼각분포 유지 |
| 연간 고정 운영비 | 정규분포를 쓰면 드물게 음수가 나올 수 있음 | 0보다 작으면 다시 샘플링하거나 하한 설정 |
| 성과 하락률 | 범위만 알기 때문에 균등분포 적절 | 0~0.2 사이로 제한 |
| 빠진 변수 | 중도탈락률, 물가상승률, 행정 지연 가능성 | 선택 변수로 추가 가능 |

**수업에서는 단순성을 위해 4개 입력 변수만 사용한다.**


### 2단계. 시뮬레이션 코드 요청

이제 AI에게 코드를 만들게 한다. 학생은 코드를 외우지 않고, 변수 정의와 결과 해석을 점검한다.

**AI에게 이렇게 질문하세요**

```text
위 입력 변수 4개로 Python 몬테카를로 시뮬레이션 코드를 만들어줘.
조건:
- N = 10,000
- 기간 = 5년
- 5년 누적 순편익 = 매년 참여자 수 × 1인당 순편익 × (1 - 성과 하락률) - 연간 고정 운영비 를 5년 합산
- 금액 단위는 억 원으로 출력
- 평균, 5%, 50%, 95% 분위수, 손실 확률 P(순편익 < 0)을 표로 출력
- 결과 분포 히스토그램을 흑백으로 그려줘
- 코드에는 초보자가 바꿔야 할 입력값 부분을 주석으로 표시해줘
```


In [ ]:
# AI가 만들어준 시뮬레이션 코드를 여기에 붙여넣고 실행하세요.


### 3단계. 결과 분포 해석

결과가 나오면 평균보다 먼저 **위험 쪽 꼬리**를 읽는다.

| 지표 | 읽는 법 | 의사결정 의미 |
|---|---|---|
| 평균 | 장기적으로 기대되는 중심값 | 사업의 평균 매력도 |
| 5% 분위수 | 매우 나쁜 경우의 결과 | 최악 근처 손실 감내 가능성 |
| 95% 분위수 | 매우 좋은 경우의 결과 | 상방 가능성 |
| 손실 확률 | 결과가 0보다 작을 비율 | 확대·보류·재설계 판단 기준 |

**판단 기준 예시**

| 손실 확률 | 권고 |
|---:|---|
| 5% 미만 | Go |
| 5~20% | Hold: 추가 정보 수집 또는 조건부 확대 |
| 20% 초과 | Redesign: 구조 재설계 후 재검토 |

임계값은 분석가가 임의로 정하는 것이 아니라 **의사결정자의 위험 허용 수준**에 맞춰 정한다.


### 4단계. 핵심 리스크 변수 찾기

몬테카를로 결과가 위험하다고 끝내면 안 된다. 어떤 입력이 결과를 가장 흔드는지 찾아야 개선 행동이 나온다.

**AI에게 이렇게 질문하세요**

```text
방금 만든 시뮬레이션에 민감도 분석을 추가해줘.
각 입력 변수와 5년 누적 순편익의 상관계수를 계산하고,
절댓값이 큰 순서로 표를 만들어줘.
그리고 토네이도 차트를 흑백으로 그려줘.
마지막에 "가장 먼저 관리해야 할 변수 1개"와 이유를 한 문장으로 써줘.
```


In [ ]:
# AI가 만들어준 민감도 분석 코드를 여기에 붙여넣고 실행하세요.


### 5단계. 시나리오 × 시뮬레이션 통합

7주차에서 만든 시나리오 플래닝은 정성적 미래를 만든다. 9주차에서는 각 시나리오마다 입력 분포를 다르게 두고 결과 분포를 비교한다.

| 시나리오 | 참여자 수 | 1인당 순편익 | 운영비 | 해석 |
|---|---|---|---|---|
| 낙관 | 높음 | 높음 | 보통 | 확장 유리 |
| 기준 | 보통 | 보통 | 보통 | 기준 판단 |
| 비관 | 낮음 | 낮음 | 높음 | 손실 위험 점검 |
| 구조 변화 | 보통 | 편차 큼 | 높음 | 불확실성 확대 |

**AI에게 이렇게 질문하세요**

```text
위 시뮬레이션을 4개 시나리오로 확장해줘.
각 시나리오마다 입력 분포의 파라미터를 다르게 두고,
시나리오별 평균, 5%, 95%, 손실 확률을 한 표로 비교해줘.
또한 같은 그림에 4개 결과 분포를 흑백 선 그래프로 비교해줘.
```


---

## 4. 실습: 나만의 시뮬레이션

관심 있는 정책 또는 사업의 의사결정을 하나 고른다.

**예시 주제**

| 주제 | 결과 변수 | 입력 변수 예시 |
|---|---|---|
| 지역 관광 축제 | 순수익 | 방문객 수, 1인 소비액, 운영비, 우천 확률 |
| 구독 서비스 출시 | 1년 이익 | 가입자 수, 이탈률, 월 구독료, 마케팅비 |
| 공공 돌봄 앱 | 순편익 | 이용자 수, 만족도 개선 효과, 개발비, 운영비 |
| 탄소 감축 사업 | 순편익 | 감축량, 탄소 가격, 설비비, 유지비 |

실습의 목표는 완벽한 모델이 아니라 **불확실성을 숫자 하나로 숨기지 않는 의사결정표**를 만드는 것이다.


### 실습 1단계. 변수와 분포 정하기

**AI에게 이렇게 질문하세요**

```text
내 의사결정 주제는 [주제명]이야.
결정 질문은 [예: 이 사업을 확대할 것인가]야.
결과 변수는 [예: 3년 누적 순이익]으로 두고 싶어.
이 결과를 계산하는 데 필요한 입력 변수 4~6개를 추천하고,
각 변수에 적절한 분포(정규/삼각/균등/로그정규)와 파라미터 초안을 제안해줘.
분포를 그렇게 고른 이유도 표에 써줘.
```


**결과**

(AI의 답변을 여기에 붙여넣으세요)


### 실습 2단계. 시뮬레이션 실행

**AI에게 이렇게 질문하세요**

```text
위 표를 사용해서 Python 몬테카를로 시뮬레이션 코드를 만들어줘.
N=10,000번 실행하고, 결과 변수의 평균, 5%, 50%, 95% 분위수, 손실 확률 또는 목표 미달 확률을 출력해줘.
히스토그램은 흑백으로 그려줘.
초보자가 입력값을 바꾸기 쉬운 형태로 코드를 작성해줘.
```


In [ ]:
# 여기에 AI가 만들어준 분석 코드를 붙여 넣으세요.


### 실습 3단계. 의사결정 권고 작성

아래 질문에 답한다.

1. 평균만 보면 어떤 결정을 내리게 되는가?
2. 5% 분위수와 손실 확률을 보면 결정이 달라지는가?
3. 가장 중요한 리스크 변수는 무엇인가?
4. 지금 바로 Go / Hold / Redesign 중 무엇을 권고할 것인가?


## 5. 제출물

아래 표를 작성하여 제출한다.

| 항목 | 작성 내용 |
|---|---|
| 결정해야 할 일 | |
| 결과 변수 | |
| 입력 변수와 분포 4개 이상 | |
| 평균 결과 | |
| 5% / 50% / 95% 분위수 | |
| 손실 확률 또는 목표 미달 확률 | |
| 핵심 리스크 변수 1개 | |
| 결정 권고 (Go / Hold / Redesign) | |
| 추가로 확인할 정보 | |

## 오늘의 핵심

몬테카를로 시뮬레이션은 미래를 맞히는 도구가 아니다. **불확실한 입력이 결과를 얼마나 흔드는지 보여주고, 평균 뒤에 숨어 있는 손실 가능성을 의사결정자에게 드러내는** 도구다.
